# IA — Aula 5 — Aprendizado de Máquina Clássico
## Trabalho da semana: Filtro de Spam — Spambase (UCI)

**T5-A1 — Ambiente, carga e inspeção da base**

Objetivos desta etapa:

1. Preparar o ambiente de execução.
2. Carregar a base **Spambase (UCI)**.
3. Confirmar as dimensões esperadas: **4.601 registros e 57 atributos preditores**.
4. Inspecionar a variável-alvo.
5. Verificar distribuição das classes, valores ausentes e tipos.
6. Confirmar a existência dos atributos `george`, `650`, `hp` e `hpl`, que serão usados em uma etapa posterior do trabalho.

> Nesta etapa **não treinaremos modelos ainda**. O objetivo é garantir que a base e o pipeline experimental estejam corretos antes dos experimentos.


## 1. Configuração do ambiente

A célula abaixo instala as dependências necessárias no ambiente atual.

> Em Google Colab ou Jupyter, execute esta célula uma vez. Se os pacotes já estiverem instalados, a operação será rápida.


In [1]:
%pip install -q ucimlrepo pandas numpy scikit-learn matplotlib


## 2. Imports e configurações de reprodutibilidade

Usaremos `random_state = 42` nas próximas etapas para tornar os experimentos reproduzíveis.


In [2]:
from __future__ import annotations

import sys
import platform
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo

RANDOM_STATE = 42

print("Python:", sys.version.split()[0])
print("Plataforma:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("random_state:", RANDOM_STATE)


Python: 3.13.15
Plataforma: Linux-6.6.122+-x86_64-with-glibc2.39
NumPy: 2.1.3
Pandas: 2.2.3
random_state: 42


## 3. Carregamento da Spambase

A base será obtida pelo pacote `ucimlrepo`, usando o identificador **94** da UCI Machine Learning Repository.


In [3]:
spambase = fetch_ucirepo(id=94)

X = spambase.data.features.copy()
y = spambase.data.targets.squeeze().copy()

# Garantir rótulo inteiro (0 = legítimo, 1 = spam)
y = pd.to_numeric(y, errors="raise").astype(int)
y.name = "spam"

print("Dataset carregado com sucesso.")
print("X:", X.shape)
print("y:", y.shape)


Dataset carregado com sucesso.
X: (4601, 57)
y: (4601,)


## 4. Validação estrutural da base

Segundo o enunciado do trabalho, esperamos:

- **4.601 e-mails**
- **57 atributos numéricos**
- alvo binário: **0 = legítimo**, **1 = spam**

As asserções abaixo fazem o notebook interromper a execução caso a estrutura recebida seja diferente da esperada.


In [4]:
EXPECTED_ROWS = 4601
EXPECTED_FEATURES = 57

assert X.shape[0] == EXPECTED_ROWS, (
    f"Número inesperado de registros: {X.shape[0]} (esperado: {EXPECTED_ROWS})"
)
assert X.shape[1] == EXPECTED_FEATURES, (
    f"Número inesperado de atributos: {X.shape[1]} (esperado: {EXPECTED_FEATURES})"
)
assert len(y) == EXPECTED_ROWS, (
    f"Tamanho inesperado do alvo: {len(y)} (esperado: {EXPECTED_ROWS})"
)
assert set(y.unique()).issubset({0, 1}), (
    f"Alvo não binário como esperado. Valores encontrados: {sorted(y.unique())}"
)

print("✓ 4.601 registros confirmados")
print("✓ 57 atributos preditores confirmados")
print("✓ alvo binário confirmado")


✓ 4.601 registros confirmados
✓ 57 atributos preditores confirmados
✓ alvo binário confirmado


## 5. Primeira inspeção dos dados


In [5]:
display(X.head())

print("\nTipos dos atributos:")
display(X.dtypes.value_counts().rename("quantidade").to_frame())

print("\nPrimeiros nomes de atributos:")
print(X.columns[:15].tolist())


,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,word_freq_conference,char_freq_;,char_freq_(,char_freq_[,char_freq_!,char_freq_$,char_freq_#,capital_run_length_average,capital_run_length_longest,capital_run_length_total
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.0,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.0,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.0,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.0,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191
4,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.0,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191



Tipos dos atributos:


,quantidade
float64,55
int64,2



Primeiros nomes de atributos:
['word_freq_make', 'word_freq_address', 'word_freq_all', 'word_freq_3d', 'word_freq_our', 'word_freq_over', 'word_freq_remove', 'word_freq_internet', 'word_freq_order', 'word_freq_mail', 'word_freq_receive', 'word_freq_will', 'word_freq_people', 'word_freq_report', 'word_freq_addresses']


## 6. Distribuição da variável-alvo

Uma das primeiras verificações em classificação é entender o balanço das classes.

Nesta base, esperamos aproximadamente **39% spam** e **61% e-mails legítimos**.

Essa distribuição será importante posteriormente para:

- construir o baseline;
- usar divisão estratificada;
- interpretar a acurácia corretamente;
- avaliar o custo dos falsos positivos.


In [6]:
class_counts = y.value_counts().sort_index()
class_pct = y.value_counts(normalize=True).sort_index().mul(100)

class_summary = pd.DataFrame({
    "classe": ["legítimo", "spam"],
    "quantidade": [class_counts.get(0, 0), class_counts.get(1, 0)],
    "percentual": [class_pct.get(0, 0), class_pct.get(1, 0)],
}, index=[0, 1])

class_summary.index.name = "rotulo"
display(class_summary)

spam_rate = class_pct.get(1, np.nan)
legit_rate = class_pct.get(0, np.nan)

print(f"Legítimos: {legit_rate:.2f}%")
print(f"Spam:      {spam_rate:.2f}%")


,classe,quantidade,percentual
rotulo,,,
0,legítimo,2788,60.595523
1,spam,1813,39.404477


Legítimos: 60.60%
Spam:      39.40%


## 7. Qualidade básica dos dados

Antes de qualquer modelagem, verificamos:

- valores ausentes;
- infinitos;
- duplicatas;
- atributos constantes.

Essas verificações são simples, mas ajudam a evitar conclusões erradas nas etapas seguintes.


In [7]:
missing_total = int(X.isna().sum().sum())
inf_total = int(np.isinf(X.to_numpy(dtype=float)).sum())
duplicate_rows = int(pd.concat([X, y], axis=1).duplicated().sum())
constant_features = X.columns[X.nunique(dropna=False) <= 1].tolist()

quality_summary = pd.DataFrame({
    "checagem": [
        "valores ausentes",
        "valores infinitos",
        "linhas duplicadas (X + y)",
        "atributos constantes",
    ],
    "quantidade": [
        missing_total,
        inf_total,
        duplicate_rows,
        len(constant_features),
    ],
})

display(quality_summary)

if constant_features:
    print("Atributos constantes:", constant_features)


,checagem,quantidade
0,valores ausentes,0
1,valores infinitos,0
2,linhas duplicadas (X + y),391
3,atributos constantes,0


## 8. Estatísticas descritivas

As 57 variáveis são numéricas, mas não necessariamente estão na mesma escala.

Isso será especialmente importante no experimento com **k-NN**, porque esse algoritmo depende de distância entre observações.


In [8]:
summary = X.describe().T
summary["range"] = summary["max"] - summary["min"]

display(summary.head(15))

print("\nAtributos com maior amplitude:")
display(
    summary[["min", "max", "mean", "std", "range"]]
    .sort_values("range", ascending=False)
    .head(10)
)


,count,mean,std,min,25%,50%,75%,max,range
word_freq_make,4601.0,0.104553,0.305358,0.0,0.0,0.0,0.00,4.54,4.54
word_freq_address,4601.0,0.213015,1.290575,0.0,0.0,0.0,0.00,14.28,14.28
word_freq_all,4601.0,0.280656,0.504143,0.0,0.0,0.0,0.42,5.10,5.10
word_freq_3d,4601.0,0.065425,1.395151,0.0,0.0,0.0,0.00,42.81,42.81
word_freq_our,4601.0,0.312223,0.672513,0.0,0.0,0.0,0.38,10.00,10.00
word_freq_over,4601.0,0.095901,0.273824,0.0,0.0,0.0,0.00,5.88,5.88
word_freq_remove,4601.0,0.114208,0.391441,0.0,0.0,0.0,0.00,7.27,7.27
word_freq_internet,4601.0,0.105295,0.401071,0.0,0.0,0.0,0.00,11.11,11.11
word_freq_order,4601.0,0.090067,0.278616,0.0,0.0,0.0,0.00,5.26,5.26
word_freq_mail,4601.0,0.239413,0.644755,0.0,0.0,0.0,0.16,18.18,18.18



Atributos com maior amplitude:


,min,max,mean,std,range
capital_run_length_total,1.0,15841.000,283.289285,606.347851,15840.000
capital_run_length_longest,1.0,9989.000,52.172789,194.891310,9988.000
capital_run_length_average,1.0,1102.500,5.191515,31.729449,1101.500
word_freq_3d,0.0,42.810,0.065425,1.395151,42.810
word_freq_george,0.0,33.330,0.767305,3.367292,33.330
char_freq_!,0.0,32.478,0.269071,0.815672,32.478
word_freq_edu,0.0,22.050,0.179824,0.911119,22.050
word_freq_re,0.0,21.420,0.301224,1.011687,21.420
word_freq_hp,0.0,20.830,0.549504,1.671349,20.830
word_freq_85,0.0,20.000,0.105412,0.532260,20.000


## 9. Verificação dos quatro atributos do experimento de generalização

O item 5 do trabalho exige retirar os atributos associados a:

- `george`
- `650`
- `hp`
- `hpl`

Na UCI, os nomes completos esperados são:

- `word_freq_george`
- `word_freq_650`
- `word_freq_hp`
- `word_freq_hpl`


In [9]:
SPECIAL_FEATURES = [
    "word_freq_george",
    "word_freq_650",
    "word_freq_hp",
    "word_freq_hpl",
]

special_feature_check = pd.DataFrame({
    "atributo": SPECIAL_FEATURES,
    "presente": [feature in X.columns for feature in SPECIAL_FEATURES],
})

display(special_feature_check)

missing_special = [
    feature for feature in SPECIAL_FEATURES
    if feature not in X.columns
]

assert not missing_special, (
    "Atributos esperados não encontrados: "
    + ", ".join(missing_special)
)

print("✓ Os quatro atributos necessários para o experimento 5 estão presentes.")


,atributo,presente
0,word_freq_george,True
1,word_freq_650,True
2,word_freq_hp,True
3,word_freq_hpl,True


✓ Os quatro atributos necessários para o experimento 5 estão presentes.


## 10. Metadados do dataset

Esta célula registra informações fornecidas pelo repositório UCI. É útil para documentar a proveniência da base no notebook.


In [10]:
print("METADADOS")
print("-" * 80)
print(spambase.metadata)

print("\nVARIÁVEIS")
print("-" * 80)
display(spambase.variables)


METADADOS
--------------------------------------------------------------------------------
{'uci_id': 94, 'name': 'Spambase', 'repository_url': 'https://archive.ics.uci.edu/dataset/94/spambase', 'data_url': 'https://archive.ics.uci.edu/static/public/94/data.csv', 'abstract': 'Classifying Email as Spam or Non-Spam', 'area': 'Computer Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 4601, 'num_features': 57, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['Class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1999, 'last_updated': 'Mon Aug 28 2023', 'dataset_doi': '10.24432/C53G6X', 'creators': ['Mark Hopkins', 'Erik Reeber', 'George Forman', 'Jaap Suermondt'], 'intro_paper': None, 'additional_info': {'summary': 'The "spam" concept is diverse: advertisements for products/web sites, make money fast schemes, chain letters, pornography...\n\nThe classification ta

,name,role,type,demographic,description,units,missing_values
0,word_freq_make,Feature,Continuous,None,None,None,no
1,word_freq_address,Feature,Continuous,None,None,None,no
2,word_freq_all,Feature,Continuous,None,None,None,no
3,word_freq_3d,Feature,Continuous,None,None,None,no
4,word_freq_our,Feature,Continuous,None,None,None,no
5,word_freq_over,Feature,Continuous,None,None,None,no
6,word_freq_remove,Feature,Continuous,None,None,None,no
7,word_freq_internet,Feature,Continuous,None,None,None,no
8,word_freq_order,Feature,Continuous,None,None,None,no
9,word_freq_mail,Feature,Continuous,None,None,None,no


## 11. Checklist da T5-A1

Ao finalizar esta etapa, devemos ter confirmado:

- [x] ambiente configurado;
- [x] Spambase carregada;
- [x] 4.601 registros;
- [x] 57 atributos preditores;
- [x] alvo binário;
- [x] distribuição das classes conhecida;
- [x] qualidade básica dos dados inspecionada;
- [x] escalas dos atributos inspecionadas;
- [x] `george`, `650`, `hp` e `hpl` localizados.

### Próxima etapa — T5-A2

Construir o **baseline que classifica todo e-mail como legítimo** e calcular sua acurácia e demais métricas relevantes.


---
## T5-A2 — Baseline: sempre prever e-mail legítimo

O primeiro item do trabalho pede um **baseline que sempre prevê `"legítimo"`**.

O baseline estabelece uma referência mínima para os modelos que serão avaliados posteriormente. Como a classe majoritária da Spambase é `0 = legítimo`, um classificador que ignore completamente os atributos e sempre produza `0` terá uma acurácia igual à proporção de e-mails legítimos da base.

Entretanto, **acurácia isolada pode ser enganosa**: esse baseline não identifica nenhum spam. Por isso, além da acurácia, calcularemos a matriz de confusão, precision, recall e F1 para a classe positiva (`spam = 1`).


### 12. Imports para avaliação

Nesta etapa não há treinamento. Apenas construiremos uma predição constante e mediremos seu comportamento.


In [11]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)


### 13. Construção do baseline

O vetor `y_baseline` terá o mesmo tamanho de `y`, mas todas as previsões serão `0`, isto é, **legítimo**.


In [12]:
# Baseline: classificar todos os e-mails como legítimos (classe 0)
y_baseline = np.zeros(len(y), dtype=int)

assert len(y_baseline) == len(y)
assert set(np.unique(y_baseline)) == {0}

print(f"Total de previsões: {len(y_baseline)}")
print("Classe prevista pelo baseline: 0 = legítimo")


Total de previsões: 4601
Classe prevista pelo baseline: 0 = legítimo


### 14. Métricas do baseline

A classe positiva do problema é `1 = spam`.

Para evitar avisos ao calcular precision quando o modelo não prevê nenhum spam, utilizamos `zero_division=0`.


In [13]:
baseline_accuracy = accuracy_score(y, y_baseline)
baseline_precision = precision_score(y, y_baseline, pos_label=1, zero_division=0)
baseline_recall = recall_score(y, y_baseline, pos_label=1, zero_division=0)
baseline_f1 = f1_score(y, y_baseline, pos_label=1, zero_division=0)

baseline_results = pd.DataFrame({
    "modelo": ["Baseline — sempre legítimo"],
    "accuracy": [baseline_accuracy],
    "precision_spam": [baseline_precision],
    "recall_spam": [baseline_recall],
    "f1_spam": [baseline_f1],
})

display(
    baseline_results.style.format({
        "accuracy": "{:.4f}",
        "precision_spam": "{:.4f}",
        "recall_spam": "{:.4f}",
        "f1_spam": "{:.4f}",
    })
)

print(f"Acurácia do baseline: {baseline_accuracy:.2%}")
print(f"Recall de spam:       {baseline_recall:.2%}")


,modelo,accuracy,precision_spam,recall_spam,f1_spam
0,Baseline — sempre legítimo,0.6060,0.0000,0.0000,0.0000


Acurácia do baseline: 60.60%
Recall de spam:       0.00%


### 15. Matriz de confusão

Usaremos a convenção do `scikit-learn`:

\[
\begin{bmatrix}
TN & FP \\
FN & TP
\end{bmatrix}
\]

onde:

- **TN**: legítimo corretamente classificado como legítimo;
- **FP**: legítimo incorretamente classificado como spam;
- **FN**: spam incorretamente classificado como legítimo;
- **TP**: spam corretamente classificado como spam.


In [14]:
cm = confusion_matrix(y, y_baseline, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

confusion_df = pd.DataFrame(
    cm,
    index=["Real: legítimo (0)", "Real: spam (1)"],
    columns=["Previsto: legítimo (0)", "Previsto: spam (1)"],
)

display(confusion_df)

print(f"TN (legítimos corretamente aceitos): {tn}")
print(f"FP (legítimos marcados como spam):   {fp}")
print(f"FN (spams que passam como legítimos): {fn}")
print(f"TP (spams detectados):                {tp}")


,Previsto: legítimo (0),Previsto: spam (1)
Real: legítimo (0),2788,0
Real: spam (1),1813,0


TN (legítimos corretamente aceitos): 2788
FP (legítimos marcados como spam):   0
FN (spams que passam como legítimos): 1813
TP (spams detectados):                0


### 16. Relatório de classificação

Este relatório explicita por que a acurácia, sozinha, não é suficiente para avaliar o filtro.


In [15]:
print(
    classification_report(
        y,
        y_baseline,
        labels=[0, 1],
        target_names=["legítimo", "spam"],
        zero_division=0,
    )
)


              precision    recall  f1-score   support

    legítimo       0.61      1.00      0.75      2788
        spam       0.00      0.00      0.00      1813

    accuracy                           0.61      4601
   macro avg       0.30      0.50      0.38      4601
weighted avg       0.37      0.61      0.46      4601



### 17. Interpretação do resultado

O baseline obtém uma acurácia equivalente à proporção da classe majoritária (`legítimo`). Apesar disso, seu desempenho como **filtro de spam é nulo**, pois:

- não prevê nenhum e-mail como spam;
- `recall` da classe spam = 0;
- todos os spams são falsos negativos;
- nenhum spam é efetivamente bloqueado.

Essa etapa mostra por que **acurácia não deve ser analisada isoladamente em problemas de classificação**, especialmente quando as classes são desbalanceadas ou quando os diferentes tipos de erro possuem custos distintos.

O baseline será usado como referência mínima para os modelos avaliados nas próximas etapas.


### Checkpoint T5-A2

Ao executar esta etapa no **Google Colab**, registre os valores obtidos. Eles serão reutilizados no relatório final de uma página.

**Próxima etapa — T5-A3:** criar uma divisão **train/test estratificada**, preservar um conjunto de teste final e configurar a **validação cruzada estratificada de 5 dobras** para os modelos do trabalho.
